## INITIAL

In [ ]:
# =====================================================================
#  0. Imports and Configuration
# =====================================================================

import sys
import re
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
# Make the repo importable regardless of where the kernel was launched.
REPO_ROOT = Path(__file__).resolve().parents[3] if False else Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'SAE.py').exists():
    REPO_ROOT = REPO_ROOT.parent
ANALYSIS_DIR = REPO_ROOT / 'llm' / 'analysis'
for d in (REPO_ROOT, ANALYSIS_DIR):
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))

from doc_analysis_functions import *


from SAE_analysis_functions import (
    sae_encode_potential_batch,
    plot_coeff_matrix,
    plot_blocked_l2_histograms,
)
SAE_PARAMETERS = {
    'JUMPRELUAE_512_1e-1':{'architecture': 'JumpReLU', 'l1': '.1', 'hidden_dim': 512,  'top_K': 0, 'normalize': False},
   # 'JUMPRELUAE_512_1e-1':{'architecture': 'JumpReLU', 'l1': '1e-1', 'hidden_dim': 512,  'top_K': 0, 'normalize': False},
}

print('Available SAE keys:', list(SAE_PARAMETERS.keys()))

# =====================================================================
#  USER-FACING CONFIGURATION  – change these as needed
# =====================================================================

SAE_KEY = 'JUMPRELUAE_512_1e-1'                # key in SAE_PARAMETERS
#SAE_KEY = 'JUMPRELUAE_512_1e-1'

DATA_ROOT      = REPO_ROOT / 'datasets' / 'pile-100k'
POTENTIALS_DIR = DATA_ROOT / 'potentials' / 'pile'
#SAE_PARAMS_DIR = DATA_ROOT / 'SAE_params'
SAE_PARAMS_DIR = DATA_ROOT / 'SAE_params'

N_LOAD      = -1    # files to load per sub-directory; -1 = all
BATCH_SIZE  = 2048  # encoding batch size
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# Normalization: set per SAE key in SAE_PARAMETERS above, or override here.
# If True, input potentials are z-scored (mean=0, std=1) before encoding.
# Check args.txt in your model dir to see what was used during training.
NORMALIZE   = None   # None = use per-SAE setting; True/False = override all

# Toggles for optional analyses
DO_UMAP_AND_HEATMAP = True   # UMAP scatter + grouped codes heatmap
DO_WORD_ANALYSIS    = False   # unigram/bigram word association analysis

# Directory where all figures will be saved
FIGURES_DIR = Path('analysis_outputs') / SAE_KEY / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Anchored output dir for per-feature interpretability artifacts (.tex / .csv)
FEATURE_INTERP_DIR = ANALYSIS_DIR / 'feature_interp'
FEATURE_INTERP_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================================

params     = SAE_PARAMETERS[SAE_KEY]
ARCH       = params['architecture']
HIDDEN_DIM = params['hidden_dim']
TOP_K      = params['top_K']
MODEL_PATH = SAE_PARAMS_DIR / SAE_KEY / 'sparse_ae.pt'

print(f"SAE key      : {SAE_KEY}")
print(f"Architecture : {ARCH}")
print(f"Hidden dim   : {HIDDEN_DIM}")
print(f"Top-K        : {TOP_K}")
print(f"Model path   : {MODEL_PATH}")
print(f"Device       : {DEVICE}")
print(f"Figures dir  : {FIGURES_DIR}")


# =====================================================================
#  1. Load Potentials + Text + Metadata
# =====================================================================

EXT = '.pt'

# Discover sub-directories; fall back to a single flat directory.
subdirs = sorted([p for p in POTENTIALS_DIR.iterdir() if p.is_dir()])
if subdirs:
    subdir_file_pairs = [(s.name, sorted(s.rglob(f'*{EXT}'))) for s in subdirs]
else:
    subdir_file_pairs = [('pile', sorted(POTENTIALS_DIR.glob(f'*{EXT}')))]

print(f"Found {len(subdir_file_pairs)} sub-directory group(s):")
for name, files in subdir_file_pairs:
    print(f"  {name!r:30s}  {len(files)} files")

all_tensors = []
texts_list = []        # raw text per document (matched row-wise)
pile_set_names = []    # pile_set_name from meta dict (matched row-wise)
index = []             # list[dict] – {row_idx, subdir, filename, doc_id, path}

for subdir_name, files in subdir_file_pairs:
    n_take = len(files) if N_LOAD == -1 else min(N_LOAD, len(files))
    print(f"Loading {n_take}/{len(files)} files from {subdir_name!r} ...", flush=True)
    for p in files[:n_take]:
        d = torch.load(p, map_location='cpu', weights_only=False)
        obj = d['object']
        t   = obj if torch.is_tensor(obj) else torch.tensor(obj)
        all_tensors.append(t.float())

        # Extract text
        texts_list.append(d.get('text', ''))

        # Extract pile_set_name from meta dict
        meta = d.get('meta', {})
        if isinstance(meta, dict):
            pile_set_names.append(meta.get('pile_set_name', ''))
        else:
            pile_set_names.append('')

        # Index metadata
        m = re.search(r'doc_?(\d+)', p.stem)
        index.append({
            'row_idx' : len(all_tensors) - 1,
            'subdir'  : subdir_name,
            'filename': p.name,
            'doc_id'  : int(m.group(1)) if m else None,
            'path'    : str(p),
        })

stacked    = torch.stack(all_tensors, dim=0)   # (N, D)
INPUT_DIM  = stacked.shape[1]
str_labels = [e['subdir'] for e in index]      # keeps block order

print(f"\nStacked tensor  : {tuple(stacked.shape)}")
print(f"Input dim       : {INPUT_DIM}")
print(f"Total rows      : {len(index)}")
print(f"Texts loaded    : {sum(1 for t in texts_list if t)}")
print(f"Unique pile sets: {len(set(pile_set_names))}")

# =====================================================================
#  2. Encode with all SAEs
# =====================================================================

codes_by_sae = {}
encode_stats = {}

for sae_key, params in SAE_PARAMETERS.items():
    arch       = params['architecture']
    hidden_dim = params['hidden_dim']
    top_k      = params['top_K']
    do_normalize = NORMALIZE if NORMALIZE is not None else params.get('normalize', False)
    print(sae_key,params)
    print('normalize?', do_normalize)

    model_path = SAE_PARAMS_DIR / sae_key / 'sparse_ae.pt'

    print(f"\nEncoding {stacked.shape[0]} samples with {sae_key} on {DEVICE} ...")
    print(f"  architecture : {arch}")
    print(f"  hidden dim   : {hidden_dim}")
    print(f"  top_k        : {top_k}")
    print(f"  normalize    : {do_normalize}")
    print(f"  model path   : {model_path}")

    # Optionally z-score the input before encoding
    if do_normalize:
        mu    = stacked.mean(dim=0, keepdim=True)
        sigma = stacked.std(dim=0, keepdim=True).clamp(min=1e-6)
        encode_input = (stacked - mu) / sigma
        print(f"  Applied z-score normalization")
    else:
        encode_input = stacked

    codes = sae_encode_potential_batch(
        encode_input,
        model_path   = str(model_path),
        architecture = arch,
        m            = INPUT_DIM,
        hidden_dim   = hidden_dim,
        top_k        = top_k,
        device       = DEVICE,
        batch_size   = BATCH_SIZE,
    )

    codes_by_sae[sae_key] = codes

    sparsity = (codes == 0).float().mean().item()
    max_act = codes.max().item()
    mean_nz = codes[codes > 0].mean().item() if (codes > 0).any() else float('nan')

    encode_stats[sae_key] = {
        'shape': tuple(codes.shape),
        'sparsity': sparsity,
        'max_activation': max_act,
        'mean_nonzero': mean_nz,
    }

    print(f"  codes shape    : {tuple(codes.shape)}")
    print(f"  sparsity       : {sparsity:.3f}")
    print(f"  max activation : {max_act:.4f}")
    print(f"  mean non-zero  : {mean_nz:.4f}")

# Select codes for the active SAE key
codes = codes_by_sae[SAE_KEY]


## BASIC STATS

In [ ]:
# ── Convenience aliases ──────────────────────────────────────────────
feature_l1 = codes.abs().sum(dim=0)             # (HIDDEN_DIM,)
sorted_l1, sorted_feat_idx = torch.sort(feature_l1, descending=True)
active_count_per_sample = (codes > 0).sum(dim=1).float()
Z             = codes
order_by_mass = sorted_feat_idx
K             = HIDDEN_DIM

n_dead = (feature_l1 == 0).sum().item()
print(f"Total features : {HIDDEN_DIM}")
print(f"Dead features  : {n_dead}  ({100*n_dead/HIDDEN_DIM:.1f}%)")

# =====================================================================
#  Feature diagnostics summary
# =====================================================================

def plot_feature_diagnostics_summary(
    feature_l1,
    active_count_per_sample,
    Z,
    order_by_mass,
    FIGURES_DIR,
    K=None,
    top_n=100,
    l1_reg=None,
    suptitle=None,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    from pathlib import Path

    FIGURES_DIR = Path(FIGURES_DIR)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    if K is None:
        K = Z.shape[1]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    title_suffix = f" (l1 reg = {l1_reg})" if l1_reg is not None else ""

    # 1. Atoms ranked by total L1 mass
    sorted_l1, _ = feature_l1.sort(descending=True)
    axes[0].bar(range(K), sorted_l1.detach().cpu().numpy(), width=1.0)
    axes[0].set_yscale('log')
    axes[0].set_title(f'Atoms ranked by total L1 mass')
    axes[0].set_xlabel('Usage rank')
    axes[0].set_ylabel('Total L1 mass')

    # 2. Active features per sample
    active_counts_int = active_count_per_sample.detach().cpu().numpy().astype(int)
    count_hist = np.bincount(active_counts_int)
    xs = np.arange(len(count_hist))
    axes[1].bar(xs, count_hist, width=0.9)
    axes[1].set_title(f'Active features per sample')
    axes[1].set_xlabel('# active')
    axes[1].set_ylabel('Number of samples')

    # 3. Centered cosine similarity among top features
    TOP_N = min(top_n, K)
    top_feats = order_by_mass[:TOP_N]
    Z_top = Z[:, top_feats]

    Zc = Z_top - Z_top.mean(dim=0, keepdim=True)
    Zc = Zc / Zc.norm(dim=0, keepdim=True).clamp_min(1e-12)

    sim = Zc.T @ Zc
    sim.fill_diagonal_(0.0)

    if TOP_N > 1:
        avg_centered_cosine = sim.sum().item() / (TOP_N * (TOP_N - 1))
    else:
        avg_centered_cosine = float('nan')

    im = axes[2].imshow(sim.detach().cpu().numpy(), aspect='auto')
    axes[2].set_title(
        f'Centered cosine similarity among top-{TOP_N} features\n'
        f'Avg off-diag similarity = {avg_centered_cosine:.4f}'
    )
    axes[2].set_xlabel('Feature rank')
    axes[2].set_ylabel('Feature rank')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    if suptitle is not None:
        fig.suptitle(suptitle + title_suffix, fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 1] if suptitle is not None else None)

    fig.savefig(FIGURES_DIR / f'feature_diagnostics_summary_l1_{l1_reg}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    return avg_centered_cosine

avg_cos = plot_feature_diagnostics_summary(
    feature_l1=feature_l1,
    active_count_per_sample=active_count_per_sample,
    Z=Z,
    order_by_mass=order_by_mass,
    FIGURES_DIR=Path('pile_figures') / 'feature_stats',
    K=K,
    top_n=100,
    l1_reg=1e-1,
    suptitle='Feature diagnostics summary for JumpReLU AE',
)
print("Average centered cosine similarity:", avg_cos)

# =====================================================================
#  Two-key comparison
# =====================================================================

def plot_feature_diagnostics_summary_two_keys_first_two(
    key1,
    key2,
    codes_by_sae,
    FIGURES_DIR,
    suptitle=None,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    from pathlib import Path

    FIGURES_DIR = Path(FIGURES_DIR)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    Z1 = codes_by_sae[key1].detach()
    Z2 = codes_by_sae[key2].detach()

    _, K1 = Z1.shape
    _, K2 = Z2.shape

    feature_l1_1 = Z1.abs().sum(dim=0)
    active_count_per_sample_1 = (Z1 > 0).sum(dim=1).float()

    feature_l1_2 = Z2.abs().sum(dim=0)
    active_count_per_sample_2 = (Z2 > 0).sum(dim=1).float()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sorted_l1_1, _ = feature_l1_1.sort(descending=True)
    sorted_l1_2, _ = feature_l1_2.sort(descending=True)

    axes[0].plot(
        np.arange(K1),
        sorted_l1_1.cpu().numpy(),
        label='l1 reg = 1e-1',
        linewidth=2,
    )
    axes[0].plot(
        np.arange(K2),
        sorted_l1_2.cpu().numpy(),
        label='l1 reg = 1e-4',
        linewidth=2,
    )
    axes[0].set_yscale('log')
    axes[0].set_title('Atoms ranked by total L1 mass')
    axes[0].set_xlabel('Usage rank')
    axes[0].set_ylabel('Total L1 mass')
    axes[0].legend()

    active_counts_int_1 = active_count_per_sample_1.cpu().numpy().astype(int)
    active_counts_int_2 = active_count_per_sample_2.cpu().numpy().astype(int)

    count_hist_1 = np.bincount(active_counts_int_1)
    count_hist_2 = np.bincount(active_counts_int_2)

    L = max(len(count_hist_1), len(count_hist_2))
    count_hist_1 = np.pad(count_hist_1, (0, L - len(count_hist_1)))
    count_hist_2 = np.pad(count_hist_2, (0, L - len(count_hist_2)))
    xs = np.arange(L)

    axes[1].bar(xs, count_hist_1, width=0.45, alpha=0.6, label='l1 reg = 1e-1')
    axes[1].bar(xs, count_hist_2, width=0.45, alpha=0.6, label='l1 reg = 1e-4')
    axes[1].set_title('Active features per sample')
    axes[1].set_xlabel('# active')
    axes[1].set_ylabel('Number of samples')
    axes[1].legend()

    if suptitle is not None:
        fig.suptitle(suptitle, fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 1] if suptitle is not None else None)

    save_name = f'feature_diagnostics_first_two_{key1}_vs_{key2}.png'
    fig.savefig(FIGURES_DIR / save_name, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
'''
plot_feature_diagnostics_summary_two_keys_first_two(
    key1='JUMPRELUAE_512',
    key2='JUMPRELUAE_512_1e-4',
    codes_by_sae=codes_by_sae,
    FIGURES_DIR=Path('pile_figures') / 'feature_stats',
    suptitle='Feature diagnostics comparison',
)
'''

## CLUSTERING AND VISUALIZATION

In [ ]:
# =====================================================================
#  5. Clustering and Visualization
# =====================================================================

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

UMAP_N_NEIGHBORS  = 15
UMAP_MIN_DIST     = 0.001
UMAP_METRIC_DOCS  = 'cosine'
RANDOM_STATE      = 12

print('Config ready.')

# ---- 5.0 Feature pruning before clustering ---------------------------

codes_np_full = codes.detach().cpu().numpy()         # (N, HIDDEN_DIM)
feature_l1_np = feature_l1.detach().cpu().numpy()    # (HIDDEN_DIM,)

N, H = codes_np_full.shape
eps = 1e-12

frac_active = (codes_np_full > 0).mean(axis=0)
MIN_FRAC_ACTIVE = 0.01
MAX_FRAC_ACTIVE = 0.99
drop_frac_active = (frac_active < MIN_FRAC_ACTIVE) | (frac_active > MAX_FRAC_ACTIVE)

total_activation = codes_np_full.sum(axis=0)
BOTTOM_MASS_PERCENT = 1
positive_total_activation = total_activation[total_activation > 0]
mass_threshold = (
    np.percentile(positive_total_activation, BOTTOM_MASS_PERCENT)
    if len(positive_total_activation) > 0 else 0.0
)
drop_low_mass = total_activation <= mass_threshold

TOP_MASS_PERCENT = 0
top_mass_threshold = (
    np.percentile(positive_total_activation, 100.0 - TOP_MASS_PERCENT)
    if len(positive_total_activation) > 0 else np.inf
)
drop_high_mass = total_activation >= top_mass_threshold

drop_zero_l1 = feature_l1_np <= 0

drop_mask = drop_zero_l1 | drop_frac_active | drop_low_mass | drop_high_mass
keep_mask = ~drop_mask

kept_indices    = np.where(keep_mask)[0]
dropped_indices = np.where(drop_mask)[0]

print(f"    zero L1:         {drop_zero_l1.sum()}")
print(f"    frac_active:     {drop_frac_active.sum()}")
print(f"    low mass:        {drop_low_mass.sum()}")
print(f"    high mass:       {drop_high_mass.sum()}")
print(f"    union dropped:   {drop_mask.sum()}")

if len(kept_indices) > 0:
    print(f"  kept frac_active range: "
          f"[{frac_active[kept_indices].min():.6f}, "
          f"{frac_active[kept_indices].max():.6f}]")
    print(f"  kept total_activation range: "
          f"[{total_activation[kept_indices].min():.6e}, "
          f"{total_activation[kept_indices].max():.6e}]")

codes_np = codes_np_full[:, kept_indices]
HIDDEN_DIM_PRUNED = codes_np.shape[1]

# ---- 5.1 K-Means elbow plot for documents ---------------------------

K_range  = list(range(2, 21, 2))
inertias = []

print('Running elbow sweep ...')
for k in K_range:
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                         batch_size=4096, n_init=3)
    km.fit(codes_np)
    inertias.append(km.inertia_)
    print(f'  k={k:3d}  inertia={km.inertia_:.3e}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_range, inertias, 'o-', linewidth=1.5)
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Inertia (within-cluster SSE)')
ax.set_title(f'K-Means elbow - documents  [{SAE_KEY}]  '
             f'({HIDDEN_DIM_PRUNED} features kept)')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '5a_doc_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved: 5a_doc_elbow.png')

N_CLUSTERS_DOCS = 8  # <- change this as needed after inspecting elbow

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_range, inertias, 'o-', linewidth=1.5)
ax.axvline(N_CLUSTERS_DOCS, color='red', linestyle='--',
           label=f'chosen k={N_CLUSTERS_DOCS}')
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Inertia (within-cluster SSE)')
ax.set_title(f'K-Means elbow - documents  [{SAE_KEY}]  (chosen k={N_CLUSTERS_DOCS})')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '5a_doc_elbow_chosen.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved: 5a_doc_elbow_chosen.png')

# ---- 5.2 Document clustering: K-Means -------------------------------

codes_np_norm = codes_np / (np.linalg.norm(codes_np, axis=1, keepdims=True) + 1e-12)

print(f'Fitting K-Means (k={N_CLUSTERS_DOCS}) on normalized codes ...')
km_docs = KMeans(
    n_clusters=N_CLUSTERS_DOCS,
    random_state=RANDOM_STATE,
    n_init=10,
    max_iter=300,
)
doc_cluster_labels = km_docs.fit_predict(codes_np_norm)
print('Done. Cluster sizes:', np.bincount(doc_cluster_labels))

# ---- 5.3 UMAP scatter + codes heatmap (optional) --------------------

if DO_UMAP_AND_HEATMAP:
    import umap

    print(f'Running UMAP on normalized codes (metric={UMAP_METRIC_DOCS}) ...')
    reducer_docs = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC_DOCS,
        random_state=RANDOM_STATE,
    )
    emb_docs = reducer_docs.fit_transform(codes_np_norm)
    print('UMAP embedding shape:', emb_docs.shape)

    unique_subdirs  = sorted(set(str_labels))
    subdir_to_int   = {s: i for i, s in enumerate(unique_subdirs)}
    subdir_int_arr  = np.array([subdir_to_int[s] for s in str_labels])
    N_SUBDIRS       = len(unique_subdirs)

    cmap_clusters = cm.get_cmap('tab20', N_CLUSTERS_DOCS)
    cmap_subdirs  = cm.get_cmap('Set1',  N_SUBDIRS)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sc0 = axes[0].scatter(
        emb_docs[:, 0], emb_docs[:, 1],
        c=doc_cluster_labels, cmap=cmap_clusters,
        s=3, alpha=0.5, linewidths=0,
    )
    for c in range(N_CLUSTERS_DOCS):
        mask = doc_cluster_labels == c
        cx, cy = emb_docs[mask, 0].mean(), emb_docs[mask, 1].mean()
        axes[0].text(cx, cy, str(c), fontsize=9, fontweight='bold',
                     ha='center', va='center',
                     bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    plt.colorbar(sc0, ax=axes[0], ticks=range(N_CLUSTERS_DOCS))
    axes[0].set_title(f'Documents - K-Means clusters (k={N_CLUSTERS_DOCS})')
    axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')

    sc1 = axes[1].scatter(
        emb_docs[:, 0], emb_docs[:, 1],
        c=subdir_int_arr, cmap=cmap_subdirs,
        s=3, alpha=0.5, linewidths=0,
    )
    legend_handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=cmap_subdirs(i / max(N_SUBDIRS - 1, 1)),
                   markersize=6, label=s)
        for i, s in enumerate(unique_subdirs)
    ]
    axes[1].legend(handles=legend_handles, title='Subdirectory',
                   fontsize=7, title_fontsize=8, loc='best')
    axes[1].set_title('Documents - subdirectory origin')
    axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')

    fig.suptitle(f'Document UMAP  [{SAE_KEY}]', fontsize=13)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '5b_doc_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved: 5b_doc_umap.png')

    # Codes heatmap, grouped by document cluster
    row_order              = np.argsort(doc_cluster_labels)
    codes_grouped          = codes_np[row_order]
    cluster_labels_grouped = doc_cluster_labels[row_order]

    kept_l1 = feature_l1_np[kept_indices]
    feat_order_pruned = np.argsort(kept_l1)[::-1]
    codes_grouped = codes_grouped[:, feat_order_pruned]

    cluster_counts     = np.bincount(cluster_labels_grouped, minlength=N_CLUSTERS_DOCS)
    cluster_boundaries = np.cumsum(cluster_counts)[:-1]

    plot_vals = np.log1p(codes_grouped)
    vmax = np.percentile(plot_vals, 99)
    vmin = 0.0

    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.imshow(
        plot_vals, aspect='auto', cmap='viridis',
        interpolation='nearest', vmin=vmin, vmax=vmax,
    )
    plt.colorbar(im, ax=ax, label='log(1 + code value)')
    ax.set_xlabel('Atom (ranked by total activation)')
    ax.set_ylabel('Documents (grouped by cluster)')
    ax.set_title(
        f'SAE codes grouped by document cluster  [{SAE_KEY}]  '
        f'({HIDDEN_DIM_PRUNED} atoms kept)'
    )
    for b in cluster_boundaries:
        ax.axhline(b - 0.5, color='white', linewidth=1.0)

    cluster_starts  = np.r_[0, cluster_boundaries]
    cluster_ends    = np.r_[cluster_boundaries, len(cluster_labels_grouped)]
    cluster_centers = 0.5 * (cluster_starts + cluster_ends - 1)

    ax.set_yticks(cluster_centers)
    ax.set_yticklabels([f'C{c} (n={cluster_counts[c]})' for c in range(N_CLUSTERS_DOCS)])
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '5c_codes_grouped_by_cluster.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved: 5c_codes_grouped_by_cluster.png')
else:
    print('DO_UMAP_AND_HEATMAP = False, skipping UMAP scatter and codes heatmap.')


## TOP DOCS PER FEATURE

In [ ]:
# =====================================================================
#  Top-100 Activating Documents per Feature
# =====================================================================

TOP_N = 100

k = min(TOP_N, codes.shape[0])
topk_result = torch.topk(codes, k=k, dim=0, largest=True, sorted=True)

top_indices_per_feature = topk_result.indices   # (TOP_N, HIDDEN_DIM)
top_values_per_feature  = topk_result.values    # (TOP_N, HIDDEN_DIM)

print(f"top_indices_per_feature shape : {tuple(top_indices_per_feature.shape)}")

# Row-level lookup: row index -> metadata
row_to_meta = {e['row_idx']: e for e in index}


def inspect_feature(
    feature_idx: int,
    n_show: int = 5,
    snippet_len: int = 400,
) -> None:
    feat_rank = (sorted_feat_idx == feature_idx).nonzero(as_tuple=True)[0]
    feat_rank = feat_rank.item() + 1 if feat_rank.numel() > 0 else '?'
    total_act = feature_l1[feature_idx].item()

    print(f"Feature {feature_idx}   (activation rank #{feat_rank},  total_act={total_act:.2f})")
    print('\u2500' * 70)

    rows = top_indices_per_feature[:n_show, feature_idx].tolist()
    vals = top_values_per_feature[:n_show, feature_idx].tolist()

    for rank, (row, val) in enumerate(zip(rows, vals), start=1):
        meta    = row_to_meta[row]
        doc_id  = meta['doc_id']
        text    = texts_list[row]
        snippet = (text[:snippet_len].rstrip() + ' ...') if text and len(text) > snippet_len else (text or '[no text]')

        print(f"  #{rank}  row={row}  doc_id={doc_id}  activation={val:.4f}  "
              f"[{meta['subdir']}]  pile_set={pile_set_names[row]}")
        if snippet:
            for line in snippet.split('\n')[:8]:
                print(f"      {line}")
        else:
            print('      [text not found]')
        print()

inspect_feature(sorted_feat_idx[0].item(), n_show=5)
inspect_feature(sorted_feat_idx[1].item(), n_show=5)

# ---- Build top_docs table ----

top_docs = []
for j in range(HIDDEN_DIM):
    rows = top_indices_per_feature[:, j].tolist()
    vals = top_values_per_feature[:, j].tolist()
    entries = []
    for row, val in zip(rows, vals):
        meta = row_to_meta[row]
        entries.append({
            'row_idx'       : row,
            'doc_id'        : meta['doc_id'],
            'subdir'        : meta['subdir'],
            'filename'      : meta['filename'],
            'activation'    : val,
            'pile_set_name' : pile_set_names[row],
        })
    top_docs.append(entries)

print(f"top_docs[j]  ->  list of {TOP_N} dicts for feature j")
print(f"Example entry: {top_docs[sorted_feat_idx[0].item()][0]}")

## CLUSTER-FEATURE ANALYSIS


In [ ]:
# =====================================================================
#  Cluster composition by pile_set_name
# =====================================================================

from collections import Counter
import pandas as pd

global_counts = Counter(pile_set_names)
n_total = len(pile_set_names)

rows = []
for c in range(N_CLUSTERS_DOCS):
    mask = np.where(doc_cluster_labels == c)[0]
    cluster_size = len(mask)
    cluster_labels_here = [pile_set_names[i] for i in mask]
    cluster_counts_here = Counter(cluster_labels_here)

    for label, count in cluster_counts_here.most_common():
        rows.append({
            'cluster': c,
            'cluster_size': cluster_size,
            'pile_set_name': label,
            'count': count,
            'pct_of_cluster': 100.0 * count / cluster_size,
            'pct_of_label_total': 100.0 * count / global_counts[label],
        })

df_comp = pd.DataFrame(rows)

# Display per cluster
for c in range(N_CLUSTERS_DOCS):
    sub = df_comp[df_comp['cluster'] == c].copy()
    print(f'\n{"="*75}')
    print(f'Cluster {c}  (n={sub["cluster_size"].iloc[0]})')
    print(f'{"="*75}')
    print(sub[['pile_set_name', 'count', 'pct_of_cluster', 'pct_of_label_total']].to_string(index=False))

In [ ]:
# =====================================================================
#  Feature firing matrix  (requires: codes)
# =====================================================================

codes_np = codes.detach().cpu().numpy()
N_DOCS, HIDDEN_DIM = codes_np.shape
fires = codes_np > 0   # binary: did feature fire on this document?

print(f'codes_np shape : {codes_np.shape}')
print(f'Mean sparsity  : {fires.mean():.4f}')

In [ ]:
import numpy as np
import pandas as pd

# ---- Per-cluster top-10 features with diagnostics ----------------
TOP_K = 10
ACTIVATION_THRESHOLD = 0.5

kept_indices = np.asarray(kept_indices)

if codes_np.shape[1] == len(kept_indices):
    codes_kept = codes_np
elif codes_np.shape[1] > np.max(kept_indices):
    codes_kept = codes_np[:, kept_indices]
else:
    raise ValueError(
        f"Shape mismatch: codes_np has {codes_np.shape[1]} columns, "
        f"but len(kept_indices)={len(kept_indices)} and max(kept_indices)={np.max(kept_indices)}"
    )

records = []

N = len(doc_cluster_labels)
D = codes_kept.shape[1]

mean_all = codes_kept.mean(axis=0)
per_feat_threshold = ACTIVATION_THRESHOLD * mean_all

above_thresh = codes_kept > per_feat_threshold[np.newaxis, :]
n_above_all = above_thresh.sum(axis=0)

for c in range(N_CLUSTERS_DOCS):
    in_mask = (doc_cluster_labels == c)
    out_mask = ~in_mask

    n_in = int(in_mask.sum())
    n_out = int(out_mask.sum())
    base_rate = n_in / N if N > 0 else np.nan

    if n_in == 0:
        continue

    codes_in = codes_kept[in_mask]
    codes_out = codes_kept[out_mask]

    mean_in = codes_in.mean(axis=0)
    mean_out = codes_out.mean(axis=0) if n_out > 0 else np.full(D, np.nan)
    mean_all_feats_in = mean_in.mean()

    pct_active_in = (codes_in > 0).mean(axis=0)

    n_above_in = above_thresh[in_mask].sum(axis=0)

    p_cluster_given_active = np.divide(
        n_above_in, n_above_all,
        out=np.zeros_like(n_above_in, dtype=float),
        where=n_above_all > 0,
    )

    lift_cluster_given_active = np.divide(
        p_cluster_given_active, base_rate,
        out=np.full_like(p_cluster_given_active, np.nan, dtype=float),
        where=base_rate > 0,
    )

    top_feat_idx = np.argsort(mean_in)[::-1][:TOP_K]

    for rank, fi in enumerate(top_feat_idx, start=1):
        ratio_vs_all = np.divide(
            mean_in[fi], mean_all[fi],
            out=np.array(np.nan), where=mean_all[fi] > 0
        ).item()

        ratio_vs_out = np.divide(
            mean_in[fi], mean_out[fi],
            out=np.array(np.nan), where=(n_out > 0 and mean_out[fi] > 0)
        ).item()

        ratio_vs_all_feats = np.divide(
            mean_in[fi], mean_all_feats_in,
            out=np.array(np.nan), where=mean_all_feats_in > 0
        ).item()

        records.append({
            'cluster': c,
            'cluster_size': n_in,
            'cluster_base_rate': base_rate,
            'rank': rank,
            'feature_idx_pruned': int(fi),
            'feature_idx_original': int(kept_indices[fi]),
            'mean_activation_in_cluster': float(mean_in[fi]),
            'ratio_in_vs_all': float(ratio_vs_all),
            'ratio_in_vs_out': float(ratio_vs_out),
            'ratio_feat_vs_all_feats': float(ratio_vs_all_feats),
            'pct_active_in_cluster': float(pct_active_in[fi]),
            'lift_cluster_given_above_thresh': float(lift_cluster_given_active[fi]),
        })

df_top = pd.DataFrame(records)

for c in range(N_CLUSTERS_DOCS):
    sub = df_top[df_top['cluster'] == c].copy()
    if len(sub) == 0:
        continue

    n_c = int(sub['cluster_size'].iloc[0])
    base_rate = float(sub['cluster_base_rate'].iloc[0])

    print(f'\n{"="*75}')
    print(f'Cluster {c}  (n={n_c}, base rate={base_rate:.3f})')
    print(f'{"="*75}')
    print(f'{"Rank":>4}  {"Feat":>6}  {"MeanAct":>8}  '
          f'{"In/All":>7}  {"In/Out":>7}  {"F/AllF":>7}  '
          f'{"%Active":>8}  {"Lift":>7}')
    print('-' * 75)

    for _, row in sub.iterrows():
        print(f'{int(row["rank"]):>4}  '
              f'{int(row["feature_idx_original"]):>6}  '
              f'{row["mean_activation_in_cluster"]:>8.4f}  '
              f'{row["ratio_in_vs_all"]:>7.2f}  '
              f'{row["ratio_in_vs_out"]:>7.2f}  '
              f'{row["ratio_feat_vs_all_feats"]:>7.2f}  '
              f'{row["pct_active_in_cluster"]:>7.1%}  '
              f'{row["lift_cluster_given_above_thresh"]:>7.2f}')

print(f'\nLift: P(cluster | a > t*mu) / base rate,  t={ACTIVATION_THRESHOLD}')

# ---- Global top-20 feature-cluster pairs per statistic ----------
TOP_GLOBAL = 20

stat_definitions = {
    'ratio_in_vs_all':                 ('In/All',  'Mean activation in-cluster / global mean'),
    'ratio_in_vs_out':                 ('In/Out',  'Mean activation in-cluster / mean outside cluster'),
    'ratio_feat_vs_all_feats':         ('F/AllF',  'Feature mean in-cluster / avg of all features in-cluster'),
    'pct_active_in_cluster':           ('%Active', 'Fraction of cluster docs where feature > 0'),
    'lift_cluster_given_above_thresh': ('Lift',    f'P(cluster | a > t*mu) / base rate, t={ACTIVATION_THRESHOLD}'),
}

for col, (short_name, description) in stat_definitions.items():
    top_pairs = df_top.nlargest(TOP_GLOBAL, col)

    print(f'\n{"="*75}')
    print(f'Top {TOP_GLOBAL} feature-cluster pairs by {short_name}')
    print(f'  ({description})')
    print(f'{"="*75}')
    print(f'{"#":>3}  {"Cluster":>7}  {"Feat":>6}  {"MeanAct":>8}  '
          f'{"In/All":>7}  {"In/Out":>7}  {"F/AllF":>7}  '
          f'{"%Active":>8}  {"Lift":>7}')
    print('-' * 75)

    for i, (_, row) in enumerate(top_pairs.iterrows(), 1):
        print(f'{i:>3}  '
              f'C{int(row["cluster"]):>5}  '
              f'{int(row["feature_idx_original"]):>6}  '
              f'{row["mean_activation_in_cluster"]:>8.4f}  '
              f'{row["ratio_in_vs_all"]:>7.2f}  '
              f'{row["ratio_in_vs_out"]:>7.2f}  '
              f'{row["ratio_feat_vs_all_feats"]:>7.2f}  '
              f'{row["pct_active_in_cluster"]:>7.1%}  '
              f'{row["lift_cluster_given_above_thresh"]:>7.2f}')

In [ ]:
# =====================================================================
#  Build cluster_atoms from ratio + lift
# =====================================================================

TOP_K_FEATURES = 3

codes_kept_local = codes.detach().cpu().numpy()[:, kept_indices]
mean_all_kept    = codes_kept_local.mean(axis=0)

cluster_atoms       = {}
all_feature_indices = set()

print(f'\n{"="*90}')
print(f'{"Cluster":>7}  |  {"Top by mean act. ratio":<35}  |  Top by lift')
print(f'{"="*90}')

for c in range(N_CLUSTERS_DOCS):
    mean_in = codes_kept_local[doc_cluster_labels == c].mean(axis=0)
    ratio   = np.divide(mean_in, mean_all_kept,
                        out=np.zeros_like(mean_in),
                        where=mean_all_kept > 0)
    top_idx = np.argsort(ratio)[::-1][:TOP_K_FEATURES]
    ratio_atoms = [int(kept_indices[i]) for i in top_idx]

    sub = df_top[df_top['cluster'] == c].nlargest(TOP_K_FEATURES, 'lift_cluster_given_above_thresh')
    lift_atoms = [int(r['feature_idx_original']) for _, r in sub.iterrows()]

    seen, union = set(), []
    for a in ratio_atoms + lift_atoms:
        if a not in seen:
            seen.add(a); union.append(a)
    cluster_atoms[c] = union
    all_feature_indices.update(union)

    ratio_str = ', '.join(f'{kept_indices[i]} ({ratio[i]:.2f})' for i in top_idx)
    lift_str  = ', '.join(f'{int(r["feature_idx_original"])} ({r["lift_cluster_given_above_thresh"]:.2f})'
                          for _, r in sub.iterrows())
    print(f'{c:>7}  |  {ratio_str:<35}  |  {lift_str}')


## TOP DOCS BY CLUSTER


In [ ]:
# =====================================================================
#  Write top_docs_by_cluster.tex  (grouped by document cluster)
# =====================================================================

from collections import Counter
from pathlib import Path

N_DOCS_PER_FEATURE = 20
SNIPPET_LEN        = 600
TOP_N_SOURCES      = 100
OUT_FILE           = FEATURE_INTERP_DIR / 'top_docs_by_cluster.tex'
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)


def escape_latex(s):
    if s is None:
        return ''
    s = str(s)
    return (s.replace('\\', r'\textbackslash{}')
             .replace('&', r'\&').replace('%', r'\%').replace('$', r'\$')
             .replace('#', r'\#').replace('_', r'\_')
             .replace('{', r'\{').replace('}', r'\}')
             .replace('~', r'\textasciitilde{}')
             .replace('^', r'\textasciicircum{}'))


def source_dist_str(atom_idx, top_n=TOP_N_SOURCES):
    entries = top_docs[atom_idx][:top_n]
    sources = [e['pile_set_name'] for e in entries if e['pile_set_name']]
    if not sources:
        return '(no source info)'
    counts = Counter(sources)
    total  = len(sources)
    return ', '.join(f'{escape_latex(s)} {100*c/total:.0f}\\%' for s, c in counts.most_common())


# Console summary
COL_CLUSTER, COL_ATOM, COL_SOURCES = 9, 8, 70
sep = '-' * (COL_CLUSTER + COL_ATOM + COL_SOURCES + 6)
print(f'\n{sep}')
print(f"{'cluster':>{COL_CLUSTER}}  {'atom':<{COL_ATOM}}  {'source distribution (top docs)':<{COL_SOURCES}}")
print(sep)
for c_id in sorted(cluster_atoms):
    for i, atom in enumerate(cluster_atoms[c_id]):
        print(f"{'  '+str(c_id) if i==0 else '':>{COL_CLUSTER}}  "
              f"{atom:<{COL_ATOM}}  {source_dist_str(atom):<{COL_SOURCES}}")
    print(sep)

# LaTeX output
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    f.write('\\section*{Top documents grouped by cluster}\n\n')
    for c_id in sorted(cluster_atoms):
        f.write(f'\\subsection*{{Cluster {c_id}}}\n\n')
        for atom in cluster_atoms[c_id]:
            f.write(f'\\paragraph*{{Feature {atom}}} Sources: {source_dist_str(atom)}\n\n')
            f.write('\\begin{verbatim}\n')
            for rank, entry in enumerate(top_docs[atom][:N_DOCS_PER_FEATURE], 1):
                row_idx = entry['row_idx']
                doc_id  = entry['doc_id']
                text    = texts_list[row_idx]
                snippet = (text[:SNIPPET_LEN].rstrip() + ' ...') if text and len(text) > SNIPPET_LEN else (text or '[text not found]')
                f.write(f'#{rank}  doc_id={doc_id}  activation={entry["activation"]:.4f}  '
                        f'pile_set={entry["pile_set_name"]}  [{entry["subdir"]}]\n')
                f.write(f'{snippet}\n\n')
            f.write('\\end{verbatim}\n\n')

print(f'Saved to {OUT_FILE}')


## TOP FEATURES BY ACTIVATION


In [ ]:
# =====================================================================
#  Top features by total activation -> top_docs_by_act.tex
# =====================================================================

TOP_N_FEATURES_BY_ACT = 40
N_DOCS_PER_FEATURE    = 20
SNIPPET_LEN           = 600

total_act_per_feature = codes.sum(dim=0).cpu().numpy()
ranked_by_act = np.argsort(total_act_per_feature)[::-1][:TOP_N_FEATURES_BY_ACT]

OUT_FILE = FEATURE_INTERP_DIR / 'top_docs_by_act.tex'
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# ---- Console summary -------------------------------------------------
COL_RANK, COL_ATOM, COL_ACT, COL_SRC = 5, 6, 12, 70
sep = '-' * (COL_RANK + COL_ATOM + COL_ACT + COL_SRC + 8)
print(f'\n{sep}')
print(f"{'rank':>{COL_RANK}}  {'atom':>{COL_ATOM}}  {'total_act':>{COL_ACT}}  "
      f"{'source distribution (top docs)':<{COL_SRC}}")
print(sep)
for rank, atom in enumerate(ranked_by_act, 1):
    atom = int(atom)
    total_act = float(total_act_per_feature[atom])
    # plain-text source dist (no LaTeX escaping) for console readability
    entries = top_docs[atom][:100]
    sources = [e['pile_set_name'] for e in entries if e['pile_set_name']]
    if sources:
        cnt = Counter(sources); n = len(sources)
        src_str = ', '.join(f'{s} {100*c/n:.0f}%' for s, c in cnt.most_common())
    else:
        src_str = '(no source info)'
    print(f"{rank:>{COL_RANK}}  {atom:>{COL_ATOM}}  {total_act:>{COL_ACT}.2f}  "
          f"{src_str:<{COL_SRC}}")
print(sep)

# ---- LaTeX output ----------------------------------------------------
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    f.write('\\section*{Top features by total activation}\n\n')
    for rank, atom in enumerate(ranked_by_act, 1):
        atom = int(atom)
        total_act = float(total_act_per_feature[atom])
        f.write(f'\\subsection*{{Rank {rank} -- Feature {atom} '
                f'(total activation = {total_act:.2f})}}\n\n')
        f.write(f'Sources: {source_dist_str(atom)}\n\n')
        f.write('\\begin{verbatim}\n')
        for doc_rank, entry in enumerate(top_docs[atom][:N_DOCS_PER_FEATURE], 1):
            row_idx = entry['row_idx']
            doc_id  = entry['doc_id']
            text    = texts_list[row_idx]
            snippet = (text[:SNIPPET_LEN].rstrip() + ' ...') if text and len(text) > SNIPPET_LEN else (text or '[text not found]')
            f.write(f'#{doc_rank}  doc_id={doc_id}  activation={entry["activation"]:.4f}  '
                    f'pile_set={entry["pile_set_name"]}  [{entry["subdir"]}]\n')
            f.write(f'{snippet}\n\n')
        f.write('\\end{verbatim}\n\n')

print(f'Saved to {OUT_FILE}')
print(f'Top {TOP_N_FEATURES_BY_ACT} features by activation written.')


## TOP FEATURES BY PURITY


In [ ]:
# =====================================================================
#  Top features by purity (single-source association) -> top_docs_by_purity.tex
# =====================================================================

TOP_N_FEATURES_BY_PURITY = 40
PURITY_TOP_N_DOCS        = 100   # docs per feature used to compute source dist
MIN_DOCS_WITH_SOURCE     = 20    # require at least this many labeled docs
N_DOCS_PER_FEATURE       = 20
SNIPPET_LEN              = 600

purity_records = []
for atom in range(HIDDEN_DIM):
    entries = top_docs[atom][:PURITY_TOP_N_DOCS]
    sources = [e['pile_set_name'] for e in entries if e['pile_set_name']]
    if len(sources) < MIN_DOCS_WITH_SOURCE:
        continue
    counts = Counter(sources)
    top_src, top_cnt = counts.most_common(1)[0]
    purity = top_cnt / len(sources)
    total_act = float(codes[:, atom].sum().item())
    purity_records.append((atom, purity, top_src, len(sources), total_act))

purity_records.sort(key=lambda r: (r[1], r[4]), reverse=True)
top_pure = purity_records[:TOP_N_FEATURES_BY_PURITY]

OUT_FILE = FEATURE_INTERP_DIR / 'top_docs_by_purity.tex'
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# ---- Console summary -------------------------------------------------
COL_RANK, COL_ATOM, COL_PUR, COL_SRC_NAME, COL_ACT, COL_DIST = 5, 6, 7, 22, 10, 60
sep = '-' * (COL_RANK + COL_ATOM + COL_PUR + COL_SRC_NAME + COL_ACT + COL_DIST + 12)
print(f'\n{sep}')
print(f"{'rank':>{COL_RANK}}  {'atom':>{COL_ATOM}}  {'purity':>{COL_PUR}}  "
      f"{'top source':<{COL_SRC_NAME}}  {'total_act':>{COL_ACT}}  "
      f"{'source distribution':<{COL_DIST}}")
print(sep)
for rank, (atom, purity, top_src, n_src, total_act) in enumerate(top_pure, 1):
    entries = top_docs[atom][:PURITY_TOP_N_DOCS]
    sources = [e['pile_set_name'] for e in entries if e['pile_set_name']]
    cnt = Counter(sources); n = len(sources)
    dist_str = ', '.join(f'{s} {100*c/n:.0f}%' for s, c in cnt.most_common())
    print(f"{rank:>{COL_RANK}}  {atom:>{COL_ATOM}}  {purity:>{COL_PUR}.2f}  "
          f"{top_src[:COL_SRC_NAME]:<{COL_SRC_NAME}}  {total_act:>{COL_ACT}.2f}  "
          f"{dist_str[:COL_DIST]:<{COL_DIST}}")
print(sep)

# ---- LaTeX output ----------------------------------------------------
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    f.write('\\section*{Top features by purity (single-source association)}\n\n')
    for rank, (atom, purity, top_src, n_src, total_act) in enumerate(top_pure, 1):
        f.write(f'\\subsection*{{Rank {rank} -- Feature {atom} '
                f'(purity = {purity:.2f}, top source = {escape_latex(top_src)}, '
                f'total activation = {total_act:.2f})}}\n\n')
        f.write(f'Source distribution: {source_dist_str(atom)}\n\n')
        f.write('\\begin{verbatim}\n')
        for doc_rank, entry in enumerate(top_docs[atom][:N_DOCS_PER_FEATURE], 1):
            row_idx = entry['row_idx']
            doc_id  = entry['doc_id']
            text    = texts_list[row_idx]
            snippet = (text[:SNIPPET_LEN].rstrip() + ' ...') if text and len(text) > SNIPPET_LEN else (text or '[text not found]')
            f.write(f'#{doc_rank}  doc_id={doc_id}  activation={entry["activation"]:.4f}  '
                    f'pile_set={entry["pile_set_name"]}  [{entry["subdir"]}]\n')
            f.write(f'{snippet}\n\n')
        f.write('\\end{verbatim}\n\n')

print(f'Saved to {OUT_FILE}')
print(f'Top {len(top_pure)} features by purity written (requested {TOP_N_FEATURES_BY_PURITY}).')


## TOP DOCS PER SOURCE


In [ ]:
# =====================================================================
#  Top features per source (by purity over top-100 docs)
#  -> top_docs_for_sources.tex
# =====================================================================
#
# For each feature f, define its purity toward source S as:
#     purity(f, S) = |{d in top-100 docs of f : pile_set_name(d) == S}| / 100
#
# For each source S, rank all features by purity(f, S) descending and take the
# top TOP_N_FEATURES_PER_SOURCE features.

TOP_N_FEATURES_PER_SOURCE = 10
PURITY_TOP_N_DOCS         = 100
N_DOCS_PER_FEATURE        = 10
SNIPPET_LEN               = 600

unique_sources = sorted(set(s for s in pile_set_names if s))

# ---- Build feature x source purity matrix ---------------------------
source_to_idx = {s: i for i, s in enumerate(unique_sources)}
feature_source_purity = np.zeros((HIDDEN_DIM, len(unique_sources)), dtype=np.float32)

for atom in range(HIDDEN_DIM):
    entries = top_docs[atom][:PURITY_TOP_N_DOCS]
    for e in entries:
        s = e['pile_set_name']
        if s in source_to_idx:
            feature_source_purity[atom, source_to_idx[s]] += 1.0
feature_source_purity /= PURITY_TOP_N_DOCS

# ---- Per-source ranking of features ---------------------------------
source_top_features = {}
for source in unique_sources:
    col = feature_source_purity[:, source_to_idx[source]]
    order = np.argsort(col)[::-1][:TOP_N_FEATURES_PER_SOURCE]
    # drop features that have zero purity for this source
    order = [int(a) for a in order if col[a] > 0]
    source_top_features[source] = [(a, float(col[a])) for a in order]

OUT_FILE = FEATURE_INTERP_DIR / 'top_docs_for_sources.tex'
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# ---- Console summary ------------------------------------------------
COL_SRC, COL_FEATS = 28, 85
sep = '-' * (COL_SRC + COL_FEATS + 4)
print(f'\n{sep}')
print(f"{'source':<{COL_SRC}}  {'top features (atom: purity)':<{COL_FEATS}}")
print(sep)
for source in unique_sources:
    feats = source_top_features[source]
    if not feats:
        feats_str = '(no feature has top-100 docs from this source)'
    else:
        feats_str = ', '.join(f'{a}: {p:.2f}' for a, p in feats)
    print(f"{source[:COL_SRC]:<{COL_SRC}}  {feats_str[:COL_FEATS]:<{COL_FEATS}}")
print(sep)
print(f'Settings: top_n_features_per_source={TOP_N_FEATURES_PER_SOURCE}, '
      f'purity_top_n_docs={PURITY_TOP_N_DOCS}')

# ---- LaTeX output ---------------------------------------------------
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    f.write('\\section*{Top features per source (by purity over top-100 docs)}\n\n')
    for source in unique_sources:
        feats = source_top_features[source]
        n_source_docs = sum(1 for s in pile_set_names if s == source)
        f.write(f'\\subsection*{{{escape_latex(source)} '
                f'(n\\_docs = {n_source_docs})}}\n\n')
        if not feats:
            f.write('No feature has any of its top-100 docs from this source.\n\n')
            continue
        for rank, (atom, purity) in enumerate(feats, 1):
            f.write(f'\\paragraph*{{Rank {rank} -- Feature {atom} '
                    f'(purity = {purity:.2f})}}\n\n')
            f.write(f'Full source distribution: {source_dist_str(atom)}\n\n')
            f.write('\\begin{verbatim}\n')
            for doc_rank, entry in enumerate(top_docs[atom][:N_DOCS_PER_FEATURE], 1):
                row_idx = entry['row_idx']
                doc_id  = entry['doc_id']
                text    = texts_list[row_idx]
                snippet = (text[:SNIPPET_LEN].rstrip() + ' ...') if text and len(text) > SNIPPET_LEN else (text or '[text not found]')
                f.write(f'#{doc_rank}  doc_id={doc_id}  activation={entry["activation"]:.4f}  '
                        f'pile_set={entry["pile_set_name"]}  [{entry["subdir"]}]\n')
                f.write(f'{snippet}\n\n')
            f.write('\\end{verbatim}\n\n')

print(f'Saved to {OUT_FILE}')
print(f'Wrote top {TOP_N_FEATURES_PER_SOURCE} features for each of {len(unique_sources)} sources.')


## WORD / N-GRAM ANALYSIS (OPTIONAL)


In [ ]:
# =====================================================================
#  Word (unigram + bigram) association analysis  [optional]
# =====================================================================

if DO_WORD_ANALYSIS:
    import re
    import time
    import pandas as pd
    from scipy import sparse
    from sklearn.feature_extraction.text import CountVectorizer
    from joblib import Parallel, delayed

    # ── Build word feature matrix (unigrams + bigrams) ──────────────
    MAX_FEATURES_UNIGRAM = 2000
    MAX_FEATURES_BIGRAM  = 1000
    MIN_DF = 30
    MAX_DF = 0.10

    vec_uni = CountVectorizer(
        lowercase=True, stop_words='english', binary=True,
        ngram_range=(1, 1), min_df=MIN_DF, max_df=MAX_DF,
        max_features=MAX_FEATURES_UNIGRAM, dtype=np.uint8,
    )
    X_uni = vec_uni.fit_transform(texts_list).tocsr()

    vec_bi = CountVectorizer(
        lowercase=True, stop_words='english', binary=True,
        ngram_range=(2, 2), min_df=MIN_DF, max_df=MAX_DF,
        max_features=MAX_FEATURES_BIGRAM, dtype=np.uint8,
    )
    X_bi = vec_bi.fit_transform(texts_list).tocsr()

    X_words = sparse.hstack([X_uni, X_bi], format='csr').astype(np.uint8)
    word_names = np.array(
        [f'word={w}'   for w in vec_uni.get_feature_names_out()] +
        [f'bigram={w}' for w in vec_bi.get_feature_names_out()],
        dtype=object,
    )
    print(f'Documents     : {len(texts_list)}')
    print(f'Word features : {X_words.shape[1]}  '
          f'({X_uni.shape[1]} unigrams + {X_bi.shape[1]} bigrams)')

    # ── Per-feature word association via correlation ────────────────
    TOP_K_WORDS         = 15
    TOP_K_PRINT         = 4
    MIN_JOINT           = 5
    N_JOBS              = -1
    N_DOCS_PER_FEATURE  = 6
    SNIPPET_LEN         = 600
    WORD_ASSOC_MODE     = 'correlation'

    cluster_map = {int(f): np.array([f]) for f in all_feature_indices}
    codes_np_kept = codes.detach().cpu().numpy()[:, kept_indices]
    kept_indices_arr = np.asarray(kept_indices)

    def _correlation_words(score, X_words, word_names, min_joint=5, top_k=15):
        joint = np.asarray(X_words.T @ (score > 0).astype(np.uint8)).ravel()
        keep = joint >= min_joint
        if not keep.any():
            return None
        X_sub     = X_words[:, keep].toarray().astype(np.float32)
        names_sub = word_names[keep]
        score_z   = score - score.mean()
        score_std = score.std() + 1e-12
        word_mean = X_sub.mean(axis=0)
        word_std  = X_sub.std(axis=0) + 1e-12
        corr = ((X_sub - word_mean) * score_z[:, None]).mean(axis=0) / (word_std * score_std)
        sorted_idx = np.argsort(corr)[::-1]
        return {
            'n_pos': int((score > 0).sum()),
            'nnz':   int(keep.sum()),
            'top_positive': [(names_sub[i], float(corr[i])) for i in sorted_idx if corr[i] > 0][:top_k],
            'top_negative': [(names_sub[i], float(corr[i])) for i in sorted_idx[::-1] if corr[i] < 0][:top_k],
        }

    def _run_one(c_id, feats, codes_local, X_words, word_names, kept_local):
        feats = np.asarray(feats)
        local = np.array([np.where(kept_local == f)[0][0] for f in feats])
        score = codes_local[:, local[0]] if len(local) == 1 else codes_local[:, local].mean(axis=1)
        return c_id, feats, _correlation_words(score, X_words, word_names,
                                               min_joint=MIN_JOINT, top_k=TOP_K_WORDS)

    t0 = time.time()
    results = Parallel(n_jobs=N_JOBS, prefer='processes')(
        delayed(_run_one)(c_id, feats, codes_np_kept, X_words, word_names, kept_indices_arr)
        for c_id, feats in sorted(cluster_map.items())
    )
    cluster_results = {int(c_id): res for c_id, _, res in results}
    print(f'Word association done ({WORD_ASSOC_MODE}) in {time.time()-t0:.1f}s')

    # ── Write word-annotated top docs ────────────────────────────────
    valid = {k for k, v in cluster_results.items() if v is not None}

    def format_top_positive(atom_idx, top_k=4):
        res = cluster_results.get(atom_idx)
        if res is None or not res.get('top_positive'):
            return '(no result)'
        return ', '.join(re.sub(r'^\w+=', '', w) for w, _ in res['top_positive'][:top_k])

    OUT_FILE_WORDS = FEATURE_INTERP_DIR / f'{WORD_ASSOC_MODE}_cluster_top_docs.txt'
    OUT_FILE_WORDS.parent.mkdir(parents=True, exist_ok=True)
    with open(OUT_FILE_WORDS, 'w', encoding='utf-8') as f:
        for c_id in sorted(cluster_atoms):
            f.write(f'\n{"="*80}\nCLUSTER {c_id}\n{"="*80}\n')
            for atom in [a for a in cluster_atoms[c_id] if a in valid]:
                f.write(f'\n  -- Feature {atom}  [{format_top_positive(atom, TOP_K_PRINT)}]\n')
                f.write('  ' + '\u2500' * 60 + '\n')
                for rank, entry in enumerate(top_docs[atom][:N_DOCS_PER_FEATURE], 1):
                    row_idx = entry['row_idx']
                    doc_id  = entry['doc_id']
                    text    = texts_list[row_idx]
                    snippet = (text[:SNIPPET_LEN].rstrip() + ' ...') if text and len(text) > SNIPPET_LEN else (text or '[text not found]')
                    f.write(f'  #{rank}  doc_id={doc_id}  activation={entry["activation"]:.4f}  '
                            f'pile_set={entry["pile_set_name"]}  [{entry["subdir"]}]\n')
                    f.write(f'{snippet}\n\n')
    print(f'Saved to {OUT_FILE_WORDS}')

    # ── Top (atom, word) pairs ───────────────────────────────────────
    pair_rows = []
    for c_id, res in cluster_results.items():
        if res is None:
            continue
        for word, coef in res['top_positive']:
            word_clean = word.replace('bigram=', '').replace('word=', '')
            pair_rows.append((coef, c_id, word_clean))
    pair_rows.sort(reverse=True)

    print(f"\n{'rank':>5}  {'coef':>8}  {'atom':>8}  {'word':<35}")
    print('-' * 60)
    for rank, (coef, c_id, word) in enumerate(pair_rows[:30], 1):
        print(f"{rank:>5}  {coef:>8.3f}  {c_id:>8}  {word:<35}")

else:
    print('DO_WORD_ANALYSIS = False, skipping unigram/bigram word analysis.')


In [ ]:
import pandas as pd

def inspect_top_docs(feature_idx, top_docs, n=50, snippet_len=250):
    rows = []
    for rank, entry in enumerate(top_docs[feature_idx][:n], 1):
        row_idx = entry['row_idx']
        text = texts_list[row_idx]
        snippet = (text[:snippet_len].replace('\n', ' ').strip() if text else '[text not found]')
        rows.append({
            'rank': rank,
            'doc_id': entry['doc_id'],
            'activation': entry['activation'],
            'pile_set_name': entry['pile_set_name'],
            'subdir': entry.get('subdir', ''),
            'snippet': snippet,
            'label': '',
        })
    return pd.DataFrame(rows)

idx = 259
df_feat = inspect_top_docs(idx, top_docs, n=50)
_specific = FEATURE_INTERP_DIR / "specific_features"
_specific.mkdir(parents=True, exist_ok=True)
df_feat.to_csv(_specific / f"feature_{idx}_top50.csv", index=False)

In [ ]:
# ---- SAE atom L2 norms and coordinate means ----

import numpy as np
import matplotlib.pyplot as plt
import torch

from SAE import (
    ReLUAE, JumpReLUAE, GatedSAE, TopKAE,
    ReLUAE_monotone, TopKAE_monotone,
    JumpReLUAE_nonneg, JumpReLUAE_monotone,
)

if ARCH == 'ReLUAE':
    sae = ReLUAE(INPUT_DIM, HIDDEN_DIM)
elif ARCH == 'JumpReLU':
    sae = JumpReLUAE(INPUT_DIM, HIDDEN_DIM)
elif ARCH == 'GatedSAE':
    sae = GatedSAE(INPUT_DIM, HIDDEN_DIM)
elif ARCH == 'TopKAE':
    sae = TopKAE(INPUT_DIM, HIDDEN_DIM, top_k=TOP_K)
elif ARCH == 'ReLUAE_monotone':
    sae = ReLUAE_monotone(INPUT_DIM, HIDDEN_DIM)
elif ARCH == 'TopKAE_monotone':
    sae = TopKAE_monotone(INPUT_DIM, HIDDEN_DIM, top_k=TOP_K)
elif ARCH == 'JumpReLU_nonneg':
    sae = JumpReLUAE_nonneg(INPUT_DIM, HIDDEN_DIM)
elif ARCH == 'JumpReLU_monotone':
    sae = JumpReLUAE_monotone(INPUT_DIM, HIDDEN_DIM)
else:
    raise ValueError(f"Unknown architecture: {ARCH}")

state = torch.load(MODEL_PATH, map_location='cpu')
sae.load_state_dict(state)
sae.eval()

with torch.no_grad():
    if hasattr(sae, "atoms"):
        atoms = sae.atoms().detach().cpu().float().numpy()
    elif hasattr(sae, "W_dec"):
        atoms = sae.W_dec.detach().cpu().float().numpy().T
    elif hasattr(sae, "enc") and hasattr(sae.enc, "weight"):
        atoms = sae.enc.weight.detach().cpu().float().numpy()
    else:
        raise ValueError("Could not determine how to extract atoms from this model.")

atom_l2   = np.linalg.norm(atoms, axis=1)
atom_mean = atoms.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(atom_l2, 'o', markersize=4)
axes[0].set_title(f'Atom L2 norms')
axes[0].set_xlabel('Atom index')
axes[0].set_ylabel('L2 norm')

axes[1].plot(atom_mean, 'o', markersize=4)
axes[1].axhline(0.0, linestyle='--', linewidth=1)
axes[1].set_title(f'Average of value of atom over support')
axes[1].set_xlabel('Atom index')
axes[1].set_ylabel('Mean over coordinates')

plt.tight_layout()
plt.show()

print(f"L2 norms:    min={atom_l2.min():.4f}  mean={atom_l2.mean():.4f}  max={atom_l2.max():.4f}")
print(f"Coord means: min={atom_mean.min():.4f}  mean={atom_mean.mean():.4f}  max={atom_mean.max():.4f}")